# Serve Receive Set Prediction — Exploratory Modeling
**Project:** How predictable were we in serve receive? (2025 Season)  
**Goal:** Train a multi-class classifier to predict who will be set given rotation, pass quality, and game context.  
**Split:** Train = Sets 1–3 | Test = Set 4 | Validate = Set 5

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    log_loss,
    ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

RANDOM_STATE = 42

## 2. Load Data
CSV exported from `extract_serve_receive.R`. Each row is one serve-receive point.

In [ ]:
df = pd.read_csv("data/cleaned_plays.csv")

print(f"Total points: {len(df)}")
print(f"Sets in file: {sorted(df['set_number'].unique())}")
print()
df.head()

## 3. Quick EDA
Before modeling, get a feel for the target distribution and flag any data quality issues.

In [ ]:
# --- points per set ---
print("Points per set:")
print(df['set_number'].value_counts().sort_index())
print()

# --- special situation rates ---
print("Special situation counts:")
for flag in ['is_ace', 'is_overpass', 'out_of_system', 'no_set', 'no_attack']:
    print(f"  {flag}: {df[flag].sum()}")
print()

# --- receive quality distribution ---
print("Receive eval codes:")
print(df['receive_eval_code'].value_counts())

In [ ]:
# --- target distribution: who gets set? ---
# attacker_id is our prediction target
# exclude points with no attack (overpass, setter error, etc.)

df_model = df[df['no_attack'] == False].copy()
print(f"Rows after dropping no-attack points: {len(df_model)}")
print()
print("Attack distribution by attacker_id:")
print(df_model['attacker_id'].value_counts())

In [ ]:
# --- receiving team rotation distribution ---
# setter_position tells us which rotation the receiving team is in
# We need to decide: are we predicting for the HOME or VISITING team's receive?
# For now, let's look at both setter position columns to understand structure

print("Home setter position distribution:")
print(df_model['home_setter_position'].value_counts().sort_index())
print()
print("Visiting setter position distribution:")
print(df_model['visiting_setter_position'].value_counts().sort_index())

## 4. Feature Engineering

For this first pass we keep features simple and interpretable:
- **Rotation** of the receiving team (determines hitter options)
- **Pass quality** (receive_eval_code mapped to numeric)
- **Score differential** (game pressure)
- **Set number** (fatigue, momentum, adjustment)
- **Server ID** (serving tendencies may influence receive)

> **Note:** You'll need to decide which team is the "receiving team" for your analysis.
> Set `FOCUS_TEAM` below to the team name string as it appears in the data.

In [ ]:
# --- SET THIS to the team you want to model ---
FOCUS_TEAM = "Duke University"   # or "University of California, Berkeley"

# filter to only points where focus team is receiving
df_focus = df_model[df_model['receiving_team'] == FOCUS_TEAM].copy()
print(f"Points where {FOCUS_TEAM} is receiving: {len(df_focus)}")

In [ ]:
# --- map receive_eval_code to numeric pass quality ---
# DataVolley standard: # = perfect (4), + = positive (3), 
#                      ! = ok (2),     - = negative (1), = / overpass (0)

pass_quality_map = {
    '#': 4,
    '+': 3,
    '!': 2,
    '-': 1,
    '/': 0,
    '=': 0
}

df_focus['pass_quality'] = df_focus['receive_eval_code'].map(pass_quality_map)

# flag if any codes didn't map (worth knowing about)
unmapped = df_focus['pass_quality'].isna().sum()
if unmapped > 0:
    print(f"Warning: {unmapped} rows with unmapped receive_eval_code:")
    print(df_focus[df_focus['pass_quality'].isna()]['receive_eval_code'].value_counts())
else:
    print("All receive codes mapped successfully.")

In [ ]:
# --- derive receiving team's rotation ---
# setter_position = which rotation number the setter is in (1-6)
# home vs visiting determines which column to use

df_focus['receiving_rotation'] = np.where(
    df_focus['receiving_team'] == df_focus['home_team'],
    df_focus['home_setter_position'],
    df_focus['visiting_setter_position']
)

print("Receiving rotation distribution:")
print(df_focus['receiving_rotation'].value_counts().sort_index())

In [ ]:
# --- encode attacker_id as target label ---
# LabelEncoder converts player IDs to 0-indexed integers for XGBoost

le = LabelEncoder()
df_focus['target'] = le.fit_transform(df_focus['attacker_id'].astype(str))

print("Target classes (attacker_id -> label):")
for label, player_id in enumerate(le.classes_):
    count = (df_focus['target'] == label).sum()
    print(f"  Label {label} -> player_id {player_id}  ({count} times set)")

In [ ]:
# --- encode server_id as categorical integer ---
le_server = LabelEncoder()
df_focus['server_encoded'] = le_server.fit_transform(df_focus['server_id'].astype(str))

# --- final feature set for this first pass ---
FEATURES = [
    'receiving_rotation',  # which rotation the receiving team is in (1-6)
    'pass_quality',        # numeric pass quality (0-4)
    'score_diff',          # score differential from focus team perspective
    'set_number',          # which set of the match (1-5)
    'server_encoded',      # who is serving
]

TARGET = 'target'

# drop any rows with NaN in features
df_focus = df_focus.dropna(subset=FEATURES + [TARGET])
print(f"Final modeling rows: {len(df_focus)}")
print()
print(df_focus[FEATURES].describe())

## 5. Train / Test / Validate Split
- **Train:** Sets 1–3 (model learns tendencies)
- **Test:** Set 4 (tune and evaluate)
- **Validate:** Set 5 (final held-out check — don't peek until the end)

In [ ]:
train = df_focus[df_focus['set_number'].isin([1, 2, 3])]
test  = df_focus[df_focus['set_number'] == 4]
val   = df_focus[df_focus['set_number'] == 5]

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]
X_val,   y_val   = val[FEATURES],   val[TARGET]

print(f"Train: {len(X_train)} points (sets 1-3)")
print(f"Test:  {len(X_test)}  points (set 4)")
print(f"Val:   {len(X_val)}   points (set 5)")

## 6. Baseline Model — XGBoost
First pass with sensible defaults. We'll tune later once we confirm the pipeline runs end-to-end.

In [ ]:
model = XGBClassifier(
    n_estimators    = 100,
    max_depth       = 3,       # shallow trees — small dataset, avoids overfitting
    learning_rate   = 0.1,
    use_label_encoder = False,
    eval_metric     = 'mlogloss',
    random_state    = RANDOM_STATE
)

model.fit(X_train, y_train)

In [ ]:
# --- evaluate on test set (set 4) ---
y_pred       = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

print("=== Test Set (Set 4) Performance ===")
print()
print(classification_report(
    y_test, y_pred,
    target_names=[f"player_{c}" for c in le.classes_]
))
print(f"Log Loss: {log_loss(y_test, y_pred_proba):.4f}")

In [ ]:
# --- confusion matrix ---
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=le.classes_,
    ax=ax,
    colorbar=False
)
ax.set_title(f"Confusion Matrix — Test Set (Set 4)\n{FOCUS_TEAM}")
ax.set_xlabel("Predicted Attacker ID")
ax.set_ylabel("Actual Attacker ID")
plt.tight_layout()
plt.show()

In [ ]:
# --- feature importance ---
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 4))
importances.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title("XGBoost Feature Importance")
ax.set_xlabel("Importance Score")
plt.tight_layout()
plt.show()

In [ ]:
# --- example: probability output for a single point ---
# This is what the model would output in a live scouting context

sample = X_test.iloc[[0]]
probs  = model.predict_proba(sample)[0]

print("Example prediction for one point:")
print(f"  Features: {sample.to_dict(orient='records')[0]}")
print()
print("  Probability each attacker is set:")
for player_id, prob in zip(le.classes_, probs):
    bar = '█' * int(prob * 40)
    print(f"  player {player_id:>12}  {prob:.1%}  {bar}")

## 7. Next Steps

Once this baseline is running, the natural progression is:

1. **Multi-match training** — loop `extract_serve_receive.R` across the full season and concatenate CSVs, then use leave-one-match-out or chronological CV instead of set-based splits
2. **Richer features** — which specific hitters are *on the court* in each rotation (from the `home_player_id` / `visiting_player_id` columns), in-system vs out-of-system as a binary flag
3. **Hierarchical target** — first predict broad zone (outside / middle / right / back), then predict specific set type within zone
4. **Calibration check** — are the predicted probabilities actually trustworthy? (Brier score, reliability diagrams)
5. **Rotation-by-rotation breakdown** — where is the team most vs least predictable?